# 📄 DocuBot – AI-Powered PDF Q&A Chatbot

This notebook builds an interactive chatbot that:
- Accepts a **PDF upload** and extracts its text
- Answers questions about the document using the **Claude AI API**
- Supports **mode selection** (Detailed / Concise / Bullet Points)
- Shows a **loading spinner** while the bot processes
- Allows **saving the chat history** to a text file
- Accepts an **optional image upload** alongside questions

> Built with [Gradio](https://gradio.app) + [Anthropic Claude API](https://docs.anthropic.com)


## Step 1: Install Dependencies

In [ ]:
# Install required libraries
!pip install gradio anthropic pymupdf --quiet
print("✅ All dependencies installed!")

## Step 2: Imports & Setup

In [ ]:
import gradio as gr
import anthropic
import fitz  # PyMuPDF — used to extract text from PDF files
import base64
import os
from datetime import datetime

# ── Configuration ──────────────────────────────────────────────────────────────
# Set your Anthropic API key here, or store it as a Colab secret
# To add a secret in Colab: click the 🔑 key icon on the left sidebar
try:
    from google.colab import userdata
    API_KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    API_KEY = os.environ.get("ANTHROPIC_API_KEY", "your-api-key-here")

client = anthropic.Anthropic(api_key=API_KEY)
print("✅ Anthropic client initialized")

## Step 3: Core Logic

In [ ]:
# ── PDF Text Extraction ────────────────────────────────────────────────────────
def extract_pdf_text(pdf_path):
    """Extract all text from a PDF file using PyMuPDF."""
    if pdf_path is None:
        return ""
    doc = fitz.open(pdf_path)
    text = ""
    for page_num, page in enumerate(doc):
        text += f"\n--- Page {page_num + 1} ---\n"
        text += page.get_text()
    doc.close()
    return text.strip()


# ── Image Encoding ─────────────────────────────────────────────────────────────
def encode_image(image_path):
    """Convert an image file to base64 for the Claude API."""
    if image_path is None:
        return None, None
    with open(image_path, "rb") as f:
        data = f.read()
    ext = os.path.splitext(image_path)[1].lower()
    mime = {"jpg": "image/jpeg", "jpeg": "image/jpeg",
            "png": "image/png", "gif": "image/gif",
            "webp": "image/webp"}.get(ext.lstrip("."), "image/png")
    return base64.standard_b64encode(data).decode("utf-8"), mime


# ── Response Mode Prompt Builder ───────────────────────────────────────────────
def build_system_prompt(mode):
    """Return a system prompt tuned to the chosen response mode."""
    base = (
        "You are DocuBot, a helpful AI assistant specialized in answering "
        "questions about documents. The user has uploaded a PDF, and you "
        "should answer questions based on its contents. If the answer "
        "cannot be found in the document, say so clearly."
    )
    modes = {
        "📝 Detailed":     base + " Provide thorough, well-explained answers with examples where helpful.",
        "⚡ Concise":      base + " Keep answers short and to the point — no more than 2-3 sentences.",
        "🔢 Bullet Points": base + " Always format your answers as bullet-point lists for easy scanning.",
    }
    return modes.get(mode, base)


print("✅ Core functions ready")

## Step 4: Chat Function

In [ ]:
# ── Main Chat Handler ──────────────────────────────────────────────────────────
def chat(user_message, history, pdf_file, image_file, response_mode):
    """
    Main chat function called by Gradio on every user message.
    
    Parameters
    ----------
    user_message  : str  – the user's typed question
    history       : list – previous [user, assistant] message pairs
    pdf_file      : str  – path to uploaded PDF (or None)
    image_file    : str  – path to uploaded image (or None)
    response_mode : str  – dropdown selection for answer style
    """
    if not user_message.strip():
        return history, history

    # Extract document text if a PDF was uploaded
    doc_text = extract_pdf_text(pdf_file)

    # Build the user content block
    content = []

    # Attach image if provided (extra feature: image upload)
    if image_file:
        img_b64, img_mime = encode_image(image_file)
        if img_b64:
            content.append({
                "type": "image",
                "source": {"type": "base64", "media_type": img_mime, "data": img_b64}
            })

    # Compose the text portion of the message
    if doc_text:
        full_text = (
            f"DOCUMENT CONTENTS:\n{doc_text}\n\n"
            f"USER QUESTION:\n{user_message}"
        )
    else:
        full_text = user_message + "\n\n(No document uploaded yet.)"

    content.append({"type": "text", "text": full_text})

    # Convert chat history to Anthropic message format
    messages = []
    for human, bot in history:
        messages.append({"role": "user",      "content": human})
        messages.append({"role": "assistant", "content": bot})
    messages.append({"role": "user", "content": content})

    # Call the Claude API
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        system=build_system_prompt(response_mode),
        messages=messages,
    )

    bot_reply = response.content[0].text

    # Update and return history
    history = history + [[user_message, bot_reply]]
    return history, history


# ── Save Chat History ──────────────────────────────────────────────────────────
def save_chat(history):
    """Save the current chat history to a timestamped .txt file."""
    if not history:
        return None
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"/tmp/docubot_chat_{timestamp}.txt"
    with open(filename, "w") as f:
        f.write("DocuBot Chat History\n")
        f.write("=" * 40 + "\n\n")
        for i, (user_msg, bot_msg) in enumerate(history, 1):
            f.write(f"[Q{i}] You:\n{user_msg}\n\n")
            f.write(f"[A{i}] DocuBot:\n{bot_msg}\n")
            f.write("-" * 40 + "\n\n")
    return filename


print("✅ Chat functions ready")

## Step 5: Build & Launch the Gradio UI

In [ ]:
# ── Custom CSS Styling ─────────────────────────────────────────────────────────
CSS = """
#docubot-header {
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
    padding: 24px 32px;
    border-radius: 16px;
    margin-bottom: 16px;
    text-align: center;
}
#docubot-header h1 { color: #e94560; font-size: 2.2rem; margin: 0 0 4px 0; }
#docubot-header p  { color: #a0aec0; font-size: 1rem; margin: 0; }
.upload-panel { background: #1a1a2e; border-radius: 12px; padding: 16px; }
.chat-panel   { background: #16213e; border-radius: 12px; padding: 16px; }
.gr-button.primary-btn {
    background: #e94560 !important;
    border: none !important;
    color: white !important;
    font-weight: bold !important;
}
"""

# ── Build the Interface ────────────────────────────────────────────────────────
with gr.Blocks(css=CSS, theme=gr.themes.Base(), title="DocuBot") as demo:

    # Header banner
    gr.HTML("""
    <div id="docubot-header">
        <h1>📄 DocuBot</h1>
        <p>Upload a PDF — then ask me anything about it!</p>
    </div>
    """)

    # Hidden state to hold chat history across turns
    chat_state = gr.State([])

    with gr.Row():
        # ── Left Panel: Uploads & Settings ──────────────────────────────────
        with gr.Column(scale=1, elem_classes=["upload-panel"]):
            gr.Markdown("### ⚙️ Settings")

            # Extra feature 1: Dropdown for response style
            response_mode = gr.Dropdown(
                choices=["📝 Detailed", "⚡ Concise", "🔢 Bullet Points"],
                value="📝 Detailed",
                label="Response Style",
                info="Choose how DocuBot formats its answers"
            )

            gr.Markdown("### 📎 Upload Document")
            pdf_upload = gr.File(
                label="Upload PDF",
                file_types=[".pdf"],
                type="filepath"
            )

            # Extra feature 2: Optional image upload
            gr.Markdown("### 🖼️ Attach an Image (optional)")
            image_upload = gr.Image(
                label="Upload Image",
                type="filepath",
                sources=["upload"]
            )

            # Extra feature 3: Save chat history button
            gr.Markdown("### 💾 Export")
            save_btn  = gr.Button("Save Chat History", variant="secondary")
            save_file = gr.File(label="Download Chat", visible=True)

        # ── Right Panel: Chat Interface ──────────────────────────────────────
        with gr.Column(scale=2, elem_classes=["chat-panel"]):
            gr.Markdown("### 💬 Chat")

            chatbot = gr.Chatbot(
                label="DocuBot",
                height=450,
                bubble_full_width=False,
                avatar_images=(
                    None,          # user avatar (None = default)
                    "https://api.dicebear.com/7.x/bottts/svg?seed=docubot"  # bot avatar
                ),
                show_label=False,
            )

            with gr.Row():
                user_input = gr.Textbox(
                    placeholder="Ask something about your document…",
                    label="Your question",
                    show_label=False,
                    scale=5,
                    lines=1,
                )
                send_btn  = gr.Button("Send ➤", scale=1, variant="primary",
                                      elem_classes=["primary-btn"])

            clear_btn = gr.Button("🗑️ Clear Chat", variant="secondary", size="sm")

            # Extra feature 4: Status bar (shows "Thinking…" while processing)
            status_bar = gr.Textbox(
                value="Ready",
                label="Status",
                interactive=False,
                max_lines=1,
            )

    # ── Event Wiring ───────────────────────────────────────────────────────────
    def set_thinking():
        return "🤔 Thinking…"

    def set_ready():
        return "✅ Ready"

    def clear_chat():
        return [], []

    # Send on button click
    send_btn.click(
        fn=set_thinking,
        outputs=status_bar
    ).then(
        fn=chat,
        inputs=[user_input, chat_state, pdf_upload, image_upload, response_mode],
        outputs=[chatbot, chat_state],
    ).then(
        fn=set_ready,
        outputs=status_bar
    ).then(
        fn=lambda: "",   # clear the input box after sending
        outputs=user_input
    )

    # Also trigger on Enter key
    user_input.submit(
        fn=set_thinking,
        outputs=status_bar
    ).then(
        fn=chat,
        inputs=[user_input, chat_state, pdf_upload, image_upload, response_mode],
        outputs=[chatbot, chat_state],
    ).then(
        fn=set_ready,
        outputs=status_bar
    ).then(
        fn=lambda: "",
        outputs=user_input
    )

    # Clear chat
    clear_btn.click(fn=clear_chat, outputs=[chatbot, chat_state])

    # Save chat history
    save_btn.click(fn=save_chat, inputs=chat_state, outputs=save_file)

# ── Launch ─────────────────────────────────────────────────────────────────────
demo.launch(share=True, debug=False)


---
## 💡 Reflection

### What was added beyond the tutorial?

| Feature | Description |
|---|---|
| 🔢 **Response mode dropdown** | User picks Detailed / Concise / Bullet Points — changes the system prompt dynamically |
| 🖼️ **Image upload** | User can attach an image alongside their question for visual context |
| 💾 **Save chat history** | Exports the full conversation to a timestamped `.txt` file |
| ⏳ **Status bar** | Shows `"Thinking…"` while the API is processing, then `"Ready"` when done |
| 🎨 **Custom CSS theme** | Dark gradient header, colored send button, and branded layout |

### How to use
1. Upload your PDF using the left panel
2. (Optional) attach an image for visual questions
3. Pick a response style from the dropdown
4. Type your question and press **Send** or hit **Enter**
5. Click **Save Chat History** to download the conversation
